# MendSpeech Day 1: Waveforms, Sampling Rates, and Acoustic Baselines

**MendSpeech Research Lab** • Week 1 (Audio, Degradation, and Measurement Foundations)

This notebook establishes the foundational digital signal processing (DSP) and audio ingestion pipeline for the **MendSpeech** project. We explore:
1. Loading and standardizing multi-channel/arbitrary sample rate audio into a single-channel 16 kHz PyTorch tensor.
2. Computing physical acoustic properties: **RMS Energy**, **Peak Amplitude**, **Dynamic Range**, and **Clipping Ratio**.
3. Resampling ablation and spectral analysis across **8 kHz**, **16 kHz**, **24 kHz**, and **48 kHz**.
4. Demonstrating the **Nyquist-Shannon Sampling Theorem** and explaining why 16 kHz is the de-facto speech standard.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import IPython.display as ipd

# Ensure root project modules can be imported
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.audio.loader import (
    load_audio, save_audio, resample_audio, compute_audio_metadata,
    compute_rms, rms_to_db, compute_peak, peak_to_dbfs, compute_clipping_ratio, normalize_audio
)

print(f"PyTorch version: {torch.__version__}")
print(f"MendSpeech src.audio module imported successfully.")

## 1. Clean Speech Dataset & Manifest

Let's inspect the baseline clean dataset generated in .

In [2]:
df_manifest = pd.read_csv("data/clean_manifest.csv")
df_manifest

## 2. Ingesting and Inspecting a Clean Speech Clip

We load  and inspect its shape and acoustic statistics.

In [3]:
clip_path = "data/clean_16k/clean_01.wav"
waveform, sr = load_audio(clip_path, target_sr=16000, mono=True)

print(f"Waveform Tensor Shape: {waveform.shape} [channels, samples]")
print(f"Sample Rate: {sr} Hz")
print(f"Duration: {waveform.shape[1] / sr:.2f} seconds")
print(f"RMS Energy: {compute_rms(waveform):.4f} ({rms_to_db(compute_rms(waveform)):.2f} dBFS)")
print(f"Peak Amplitude: {compute_peak(waveform):.4f} ({peak_to_dbfs(compute_peak(waveform)):.2f} dBFS)")
print(f"Clipping Ratio: {compute_clipping_ratio(waveform):.4f}")

# Audio playback widget
ipd.Audio(waveform.squeeze().numpy(), rate=sr)

## 3. Sampling Rate & Nyquist Spectral Ablation

We compare the time-domain waveform and frequency-domain magnitude spectrum across **8,000 Hz**, **16,000 Hz**, **24,000 Hz**, and **48,000 Hz**.

In [4]:
sample_rates = [8000, 16000, 24000, 48000]
fig, axs = plt.subplots(len(sample_rates), 2, figsize=(14, 10))
fig.suptitle("Waveform & FFT Magnitude Spectrum Across Sampling Rates", fontsize=14, fontweight="bold")

for idx, target_sr in enumerate(sample_rates):
    resampled = resample_audio(waveform, orig_sr=16000, target_sr=target_sr)
    sig = resampled.squeeze().numpy()
    time_axis = np.linspace(0, len(sig) / target_sr, len(sig))
    
    # Time Domain (First 2.0 seconds)
    max_t = int(2.0 * target_sr)
    axs[idx, 0].plot(time_axis[:max_t], sig[:max_t], color="#1f77b4", lw=0.8)
    axs[idx, 0].set_title(f"{target_sr} Hz (Nyquist: {target_sr//2} Hz) - Time Domain", fontsize=10)
    axs[idx, 0].set_ylabel("Amplitude")
    axs[idx, 0].set_ylim(-1.0, 1.0)
    axs[idx, 0].grid(True, alpha=0.3)
    
    # Frequency Domain (FFT Magnitude Spectrum)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / target_sr)
    mag_db = 20 * np.log10(np.abs(np.fft.rfft(sig)) + 1e-9)
    mag_db_norm = mag_db - np.max(mag_db)
    
    axs[idx, 1].plot(freqs, mag_db_norm, color="#d62728", lw=0.8)
    axs[idx, 1].axvline(target_sr // 2, color="black", linestyle="--", alpha=0.7, label=f"Nyquist: {target_sr//2} Hz")
    axs[idx, 1].set_title(f"{target_sr} Hz - FFT Magnitude Spectrum", fontsize=10)
    axs[idx, 1].set_ylabel("Magnitude (dB)")
    axs[idx, 1].set_xlim(0, 24000)
    axs[idx, 1].set_ylim(-80, 5)
    axs[idx, 1].grid(True, alpha=0.3)
    axs[idx, 1].legend(loc="upper right", fontsize=8)

axs[-1, 0].set_xlabel("Time (seconds)")
axs[-1, 1].set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

## 4. Resampling Metric Tradeoff Table

Let's display the measured file sizes, sample counts, and RMS energies across sample rates.

In [5]:
df_resample = pd.read_csv("results/day01_resampling_comparison_table.csv")
df_resample

## 5. Summary & Day 1 Key Findings

- **8 kHz (Telephony):** Low pass filtered at 4 kHz. Voiceless fricatives (, ) lose distinct high-frequency energy.
- **16 kHz (Speech Standard):** Nyquist cutoff at 8 kHz captures virtually all phonetic formants ($ through $) and sibilance with zero perceptible loss in speech recognition intelligibility.
- **48 kHz (Studio Audio):** Triples the tensor size and compute burden without providing additional phonetic discrimination value for ASR.
- **Conclusion:** MendSpeech pipelines will standardize on **16 kHz Mono Float32** tensors normalized to hBc20	ext{ dBFS}$ with headroom limiting.